In [ ]:
df = pd.read_csv("ALL_SAHAM_RELEVANCY_NEWS.csv")
df

In [ ]:
df=df.drop(columns=["Unnamed: 0"])
df.head()

In [ ]:
df.isna().sum()

In [ ]:
!pip install -q transformers torch pandas

import pandas as pd
import re
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification

# load model
MODEL_ID = "taufiqdp/indonesian-sentiment"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID)

print("Label mapping (id2label):", model.config.id2label)

device = 0 if torch.cuda.is_available() else -1
sentiment_pipe = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=device,
    return_all_scores=False
)

In [ ]:
def predict_batch(texts, batch_size=32):
    results = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        preds = sentiment_pipe(batch, truncation=True, max_length=512)
        results.extend(preds)
    return results

In [ ]:
positif_keywords = ["menguat", "rebound", "bullish", "surplus", "naik", "melonjak", "penguatan", "stabil", "optimis", "pulih", "rekor", "melambung"]
negatif_keywords = ["melemah", "anjlok", "bearish", "defisit", "turun", "merosot", "penurunan", "tertekan", "resesi", "krisis", "minus", "collapse", "pelemahan"]

positif_pattern = re.compile(r"\b(" + "|".join(positif_keywords) + r")\b", re.IGNORECASE)
negatif_pattern = re.compile(r"\b(" + "|".join(negatif_keywords) + r")\b", re.IGNORECASE)

def apply_keyword_override(text, model_label, model_score):
    pos_match = bool(positif_pattern.search(text))
    neg_match = bool(negatif_pattern.search(text))
    if pos_match and not neg_match:
        return "positif", 1.0
    elif neg_match and not pos_match:
        return "negatif", 1.0
    elif pos_match and neg_match:
        pos_count = len(positif_pattern.findall(text))
        neg_count = len(negatif_pattern.findall(text))
        if pos_count > neg_count: return "positif", 1.0
        elif neg_count > pos_count: return "negatif", 1.0
    return model_label, model_score

In [ ]:
df["konten_clean"] = df["konten_clean"].fillna("").astype(str)
preds = predict_batch(df["konten_clean"].tolist(), batch_size=32)
df["sentiment_label_model"] = [p["label"] for p in preds]
df["sentiment_score_model"] = [float(p["score"]) for p in preds]
mapping = {'negatif': 'negatif', 'netral': 'netral', 'positif': 'positif'}
df["sentiment_label_model_id"] = df["sentiment_label_model"].map(mapping).fillna(df["sentiment_label_model"])
adjusted = [apply_keyword_override(t, l, s) for t, l, s in zip(df["konten_clean"], df["sentiment_label_model_id"], df["sentiment_score_model"])]
df["sentiment_label"], df["sentiment_score"] = zip(*adjusted)

In [ ]:
df_final = df[["tanggal", "kategori", "judul", "url", "konten_clean", "sentiment_label", "sentiment_score"]]
df_final.to_csv("df_sentiment_saham_final.csv", index=False)
print("Done!")